In [ ]:
import pandas as pd

DATA = '../data/case-study/processed'

# load both tests
hr_psychometric = pd.read_csv(f'{DATA}/hr_01.csv')
hr_baseline = pd.read_csv(f'{DATA}/hr.csv')

# filter high confidence
hr_psychometric_high_conf = hr_psychometric[hr_psychometric['confidence'] == 1.0]
hr_baseline_high_conf = hr_baseline[hr_baseline['confidence'] == 1.0]

# overall avg + std
overall_avg_hr_psychometric = hr_psychometric_high_conf['heart_rate'].mean()
overall_sd_hr_psychometric = hr_psychometric_high_conf['heart_rate'].std()

overall_avg_hr_baseline = hr_baseline_high_conf['heart_rate'].mean()
overall_sd_hr_baseline = hr_baseline_high_conf['heart_rate'].std()

# create table
combined_stats_df = pd.DataFrame({
    'Test': ['Psychometric Test', 'Baseline Test'],
    'Average HR': [overall_avg_hr_psychometric, overall_avg_hr_baseline],
    'Standard Deviation HR': [overall_sd_hr_psychometric, overall_sd_hr_baseline]
})

print(combined_stats_df)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

DATA = '../data/case-study/processed'

# load all sessions
hr_baseline = pd.read_csv(f'{DATA}/hr.csv')
hr_01 = pd.read_csv(f'{DATA}/hr_01.csv')
hr_02 = pd.read_csv(f'{DATA}/hr_02.csv')
hr_03 = pd.read_csv(f'{DATA}/hr_03.csv')

sessions = {
    'Baseline': hr_baseline[hr_baseline['confidence'] == 1.0]['heart_rate'],
    'Session 01': hr_01[hr_01['confidence'] == 1.0]['heart_rate'],
    'Session 02': hr_02[hr_02['confidence'] == 1.0]['heart_rate'],
    'Session 03': hr_03[hr_03['confidence'] == 1.0]['heart_rate']
}

# descriptive stats
stats_table = pd.DataFrame({
    name: {
        'N': len(data),
        'Mean': data.mean(),
        'SD': data.std(),
        'Median': data.median(),
        'IQR': data.quantile(0.75) - data.quantile(0.25),
        'Min': data.min(),
        'Max': data.max()
    }
    for name, data in sessions.items()
}).T.round(2)

print("=== Descriptive Statistics ===\n")
print(stats_table.to_string())

# Kruskal-Wallis test
print("\n=== Kruskal-Wallis Test (all sessions) ===\n")
h_stat, p_val = stats.kruskal(*sessions.values())
print(f"H={h_stat:.3f}, p={p_val:.4f}")

# pairwise Mann-Whitney U
print("\n=== Pairwise Mann-Whitney U Tests ===\n")
names = list(sessions.keys())
from itertools import combinations
p_vals = []
pair_labels = []
for i, j in combinations(range(len(names)), 2):
    u_stat, p = stats.mannwhitneyu(sessions[names[i]], sessions[names[j]], alternative='two-sided')
    p_vals.append(p)
    pair_labels.append(f"{names[i]} vs {names[j]}")

# Holm correction
sorted_idx = np.argsort(p_vals)
corrected = np.zeros(len(p_vals))
m = len(p_vals)
for rank, idx in enumerate(sorted_idx):
    corrected[idx] = min(p_vals[idx] * (m - rank), 1.0)

for k in range(len(pair_labels)):
    sig = "*" if corrected[k] < 0.05 else "ns"
    print(f"  {pair_labels[k]}: U p={p_vals[k]:.4f}, corrected={corrected[k]:.4f} {sig}")